In [1]:
%matplotlib inline

# paired_ttest_corrected_resampled: Corrected resampled paired *t* test

Corrected resampled paired *t* test (Nadeau and Bengio) to compare the performance of two models

> `from mlxtend.evaluate import paired_ttest_corrected_resampled`    

## Overview

The [resampled paired *t* test](paired_ttest_resampled.md) repeats a random train/test split $J$ times, fits two models A and B on each training set, and records the difference in their test scores, $p^{(j)} = p^{(j)}_A - p^{(j)}_B$. It then treats the $J$ differences as independent samples. They are not: the training sets of different splits overlap heavily, and so do the test sets. As a result, the variance of $\overline{p}$ is underestimated and the test reports a "significant" difference much more often than the chosen significance level allows [1, 2].

Nadeau and Bengio [2] proposed a simple correction that accounts for this overlap. With $n_1$ training examples and $n_2$ test examples per split, the variance factor $\frac{1}{J}$ in the *t* statistic is replaced by $\frac{1}{J} + \frac{n_2}{n_1}$:

$$t = \frac{\overline{p}}{\sqrt{\left(\frac{1}{J} + \frac{n_2}{n_1}\right) \hat{\sigma}^2}}, \quad \hat{\sigma}^2 = \frac{1}{J-1} \sum_{j=1}^{J} \left(p^{(j)} - \overline{p}\right)^2,$$

where $\overline{p} = \frac{1}{J} \sum_{j=1}^{J} p^{(j)}$. As in the uncorrected test, the p value is computed from a *t* distribution with $J-1$ degrees of freedom.

Without the correction, the denominator would be $\sqrt{\hat{\sigma}^2 / J}$, which is the standard resampled paired *t* test. Because $\frac{n_2}{n_1}$ does not shrink as $J$ grows, adding more splits no longer makes arbitrarily small differences look significant.

The same correction applies to repeated *k*-fold cross-validation, where $J = k \cdot r$ for $r$ repetitions and $\frac{n_2}{n_1} = \frac{1}{k-1}$. Bouckaert and Frank [3] call this the *corrected repeated k-fold cv test* and recommend it with 10 repetitions of 10-fold cross-validation.

**When to use it.** Use this test when you want to compare two models with many random train/test splits or with repeated (stratified) *k*-fold cross-validation, for example because your evaluation protocol already uses one of these schemes. Unlike [`paired_ttest_5x2cv`](paired_ttest_5x2cv.md) and [`combined_ftest_5x2cv`](combined_ftest_5x2cv.md), it is not tied to five repetitions of 2-fold cross-validation, so each model is trained on more data.

**Avoiding data leakage.** Both estimators are cloned and refit on the training part of every split, and the estimators that are passed in are not modified. Preprocessing steps such as feature scaling or feature selection should be part of a scikit-learn `Pipeline` so that they are also fit on the training part only.

### References

- [1] Dietterich TG (1998) Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms. *Neural Comput* 10:1895–1923.
- [2] Nadeau C, Bengio Y (2003) Inference for the Generalization Error. *Machine Learning* 52:239–281.
- [3] Bouckaert RR, Frank E (2004) Evaluating the Replicability of Significance Tests for Comparing Learning Algorithms. In: *Advances in Knowledge Discovery and Data Mining (PAKDD 2004)*, pp. 3–12.

## Example 1 - Corrected vs. uncorrected resampled paired *t* test

Assume we want to compare two classification algorithms, logistic regression and a decision tree algorithm, on the Iris dataset:

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from mlxtend.data import iris_data


X, y = iris_data()
clf1 = LogisticRegression(random_state=1, max_iter=1000)
clf2 = DecisionTreeClassifier(random_state=1)

For the same `random_seed`, `paired_ttest_corrected_resampled` uses exactly the same 30 train/test splits (70% training, 30% test data) as `paired_ttest_resampled`, so the only difference between the two results below is the variance correction:

In [3]:
from mlxtend.evaluate import paired_ttest_resampled
from mlxtend.evaluate import paired_ttest_corrected_resampled


t, p = paired_ttest_resampled(estimator1=clf1,
                              estimator2=clf2,
                              X=X, y=y,
                              random_seed=1)

print('Uncorrected resampled t test')
print('t statistic: %.3f' % t)
print('p value: %.3f\n' % p)

t, p = paired_ttest_corrected_resampled(estimator1=clf1,
                                        estimator2=clf2,
                                        X=X, y=y,
                                        random_seed=1)

print('Corrected resampled t test')
print('t statistic: %.3f' % t)
print('p value: %.3f' % p)

Uncorrected resampled t test
t statistic: 3.616
p value: 0.001



Corrected resampled t test
t statistic: 0.971
p value: 0.339


Using a significance level of $\alpha=0.05$, the uncorrected test would lead us to conclude that the two models perform significantly differently. After correcting for the overlapping splits, $p > \alpha$, so we cannot reject the null hypothesis that both models perform equally well on this dataset.

The correction does not hide real differences. If we replace the decision tree by a decision stump (a tree of depth 1), which is a much weaker model on this dataset, the corrected test still detects the difference:

In [4]:
clf2 = DecisionTreeClassifier(random_state=1, max_depth=1)

t, p = paired_ttest_corrected_resampled(estimator1=clf1,
                                        estimator2=clf2,
                                        X=X, y=y,
                                        random_seed=1)

print('t statistic: %.3f' % t)
print('p value: %.3f' % p)

t statistic: 11.407
p value: 0.000


Since $p < \alpha$, we reject the null hypothesis and conclude that the two models perform significantly differently.

## Example 2 - Corrected repeated *k*-fold cross-validation test

Instead of random train/test splits, any scikit-learn cross-validation splitter can be passed via the `cv` parameter. If `cv` is given, `num_rounds`, `test_size`, and `random_seed` are ignored, and the ratio $n_2/n_1$ is computed from the actual split sizes.

The following example uses 10 repetitions of stratified 10-fold cross-validation, as recommended in [3]. It also shows how to keep preprocessing inside the cross-validation loop: the `StandardScaler` is part of each `Pipeline`, so it is fit on the training folds only.

In [5]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from mlxtend.data import wine_data


X, y = wine_data()

pipe1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe2 = make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=1))

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=1)

t, p = paired_ttest_corrected_resampled(estimator1=pipe1,
                                        estimator2=pipe2,
                                        X=X, y=y,
                                        cv=cv,
                                        scoring='f1_macro')

print('t statistic: %.3f' % t)
print('p value: %.3f' % p)

t statistic: -0.053
p value: 0.958


Here, $p > \alpha$ for $\alpha=0.05$, so we cannot conclude that the two pipelines perform differently in terms of macro-averaged F1 score on this dataset.

## Limitations

- The correction assumes that the variance of the performance differences is dominated by the overlap of the training and test sets. It is a heuristic rather than an exact test, and its behavior depends on the learning algorithms and the dataset [2, 3]. Its false positive rate is much closer to the chosen significance level than that of the uncorrected test, but it is not guaranteed to match it exactly.
- Like all *t* tests, it assumes that the score differences are approximately normally distributed.
- The test only compares two models on one dataset. To compare models across several datasets, or to compare more than two models, other procedures are needed (see, e.g., [`cochrans_q`](cochrans_q.md) for multiple classifiers).
- If the score difference is exactly the same in every split, the variance is zero and the *t* statistic is undefined. In this case, the function returns $t = 0$ and $p = 1$ if the difference is zero, and $t = \pm\infty$ and $p = 0$ otherwise.

## API

In [6]:
with open('../../api_modules/mlxtend.evaluate/paired_ttest_corrected_resampled.md', 'r') as f:
    s = f.read()
print(s)

## paired_ttest_corrected_resampled

*paired_ttest_corrected_resampled(estimator1, estimator2, X, y, num_rounds=30, test_size=0.3, cv=None, scoring=None, random_seed=None)*

Implements the corrected resampled paired t test by
    Nadeau and Bengio (2003) to compare the performance of two models.

    The standard resampled paired t test treats the score differences
    from overlapping train/test splits as independent, which
    underestimates their variance and leads to too many false
    positives. The corrected test replaces the variance factor 1/J
    with (1/J + n_test/n_train), where J is the number of splits.
    Used with repeated k-fold splits, this is the corrected repeated
    k-fold cross-validation test by Bouckaert and Frank (2004).

**Parameters**

- `estimator1` : scikit-learn classifier or regressor



- `estimator2` : scikit-learn classifier or regressor



- `X` : {array-like, sparse matrix}, shape = [n_samples, n_features]

    Training vectors, where n_samples is t